In [ ]:
import os
from glob import glob
import random
import numpy as np
from tqdm import tqdm
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from transformers import ViTModel, ViTImageProcessor
import json
from datetime import datetime

""" Author: Puneet Sharma (UiT) with Claude Code to format and debug the pipeline """

# ============================================
# CONFIGURATION
# ============================================
backbone = "efficientnet"  # Options: "vit", "res", "convnext", "efficientnet"
DATA_FOLDER = '/local/data/Polyps_with_5_images/'
SAVE_DIR = '/local/kfold_splits/'
RANDOM_STATE = 16
N_FOLDS = 10
TEST_SPLIT = 0.2
NUM_EPOCHS = 50 
BATCH_SIZE = 32

# SimCLR hyperparameters
PROJECTION_DIM = 512
HIDDEN_DIM = 2048  # Hidden layer in projection head
TEMPERATURE = 0.5
NUM_EPOCHS_SIMCLR = 200  # Increased from 100
BATCH_SIZE_SIMCLR = 64  # 
ACCUMULATION_STEPS = 1  # Set to 2 if GPU memory limited


# Backbone-specific hyperparameters
if backbone == "vit":
    BASE_LR = 3e-4  # Much lower for ViT + AdamW
    WEIGHT_DECAY = 0.05  # Stronger regularization for ViT
    WARMUP_EPOCHS = 10  # Can increase to 20 for very large datasets
   
else:  # CNN backbones
    BASE_LR = 0.3  # Large LR for LARS
    WEIGHT_DECAY = 1e-6  # Minimal for LARS
    WARMUP_EPOCHS = 10
   
# Early Stopping
PATIENCE = 20  
MIN_DELTA = 1e-4
VAL_SPLIT = 0.2

debug_log = {
    'timestamp': datetime.now().isoformat(),
    'backbone': backbone,
    'errors': [],
    'warnings': [],
    'summary': {}
}

# Set random seeds
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using backbone: {backbone}, Device: {device}")

# ============================================
# DATA AUGMENTATION
# ============================================
simclr_transform = transforms.Compose([
    # Crop range for medical images
    transforms.RandomResizedCrop(
        size=224, 
        scale=(0.5, 1.0),  
        interpolation=transforms.InterpolationMode.BICUBIC
    ),
    
    # Both horizontal and vertical flips
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    
    # Reasonable rotation for medical images
    transforms.RandomRotation(degrees=30),
    
    # Stronger color jitter with probability
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.4,
            contrast=0.4,
            saturation=0.4,
            hue=0.1
        )
    ], p=0.8),
    
    # Random grayscale for color invariance
    transforms.RandomGrayscale(p=0.2),
    
    # Gaussian blur 
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0))
    ], p=0.5),
    
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ============================================
# SIMCLR MODEL
# ============================================
class ImprovedSimCLR(nn.Module):
    """SimCLR with projection head."""
    def __init__(self, backbone, input_dim, hidden_dim=2048, projection_dim=128):
        super().__init__()
        self.backbone = backbone
        # IMPROVED: 2-layer projection with BatchNorm
        self.projector = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, projection_dim)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        features = features.view(features.size(0), -1)
        projections = self.projector(features)
        return F.normalize(projections, dim=1)

# ============================================
# SIMCLR LOSS
# ============================================
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, z_i, z_j):
        batch_size = z_i.size(0)
        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)
        representations = torch.cat([z_i, z_j], dim=0)
        similarity_matrix = torch.matmul(representations, representations.T) / self.temperature
        mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z_i.device)
        similarity_matrix = similarity_matrix.masked_fill(mask, float('-inf'))
        labels = torch.cat([
            torch.arange(batch_size, 2 * batch_size),
            torch.arange(0, batch_size)
        ]).to(z_i.device)
        loss = F.cross_entropy(similarity_matrix, labels)
        return loss

# ============================================
# LEARNING RATE SCHEDULER
# ============================================
class WarmupCosineScheduler:
    """Warmup + cosine decay scheduler."""
    def __init__(self, optimizer, warmup_epochs, total_epochs, base_lr, min_lr=1e-5):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_epoch = 0
    
    def step(self):
        if self.current_epoch < self.warmup_epochs:
            lr = self.base_lr * (self.current_epoch + 1) / self.warmup_epochs
        else:
            progress = (self.current_epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + (self.base_lr - self.min_lr) * 0.5 * (1 + np.cos(np.pi * progress))
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        
        self.current_epoch += 1
        return lr

# ============================================
# LARS OPTIMIZER
# ============================================
class LARS(torch.optim.Optimizer):
    """Layer-wise Adaptive Rate Scaling."""
    def __init__(self, params, lr=0.1, momentum=0.9, weight_decay=1e-6, eta=0.001):
        defaults = dict(lr=lr, momentum=momentum, weight_decay=weight_decay, eta=eta)
        super().__init__(params, defaults)
        self.state_initialized = False

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            weight_decay = group['weight_decay']
            momentum = group['momentum']
            eta = group['eta']
            lr = group['lr']

            for p in group['params']:
                if p.grad is None:
                    continue

                param_norm = torch.norm(p.data)
                grad_norm = torch.norm(p.grad.data)

                if param_norm != 0 and grad_norm != 0:
                    adaptive_lr = eta * param_norm / (grad_norm + weight_decay * param_norm)
                else:
                    adaptive_lr = 1.0

                if weight_decay != 0:
                    p.grad.data.add_(p.data, alpha=weight_decay)

                if momentum != 0:
                    param_state = self.state[p]
                    if 'momentum_buffer' not in param_state:
                        buf = param_state['momentum_buffer'] = torch.clone(p.grad.data).detach()
                    else:
                        buf = param_state['momentum_buffer']
                        buf.mul_(momentum).add_(p.grad.data)
                    p.grad.data = buf

                p.data.add_(p.grad.data, alpha=-lr * adaptive_lr)

# ============================================
# MODEL INITIALIZATION
# ============================================
if backbone == "convnext":
    convnext_model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1).to(device)
    feature_extractor = torch.nn.Sequential(*list(convnext_model.children())[:-1])
    INPUT_DIM = 1024
    simclr_model = ImprovedSimCLR(feature_extractor, INPUT_DIM, HIDDEN_DIM, PROJECTION_DIM).to(device)

elif backbone == "res":
    resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(device)
    feature_extractor = torch.nn.Sequential(*list(resnet50.children())[:-1])
    INPUT_DIM = 2048
    simclr_model = ImprovedSimCLR(feature_extractor, INPUT_DIM, HIDDEN_DIM, PROJECTION_DIM).to(device)

elif backbone == "efficientnet":
    efficientnet_model = models.efficientnet_b5(weights=models.EfficientNet_B5_Weights.IMAGENET1K_V1).to(device)
    feature_extractor = torch.nn.Sequential(*list(efficientnet_model.children())[:-1])
    INPUT_DIM = 2048
    simclr_model = ImprovedSimCLR(feature_extractor, INPUT_DIM, HIDDEN_DIM, PROJECTION_DIM).to(device)

elif backbone == "vit":
    vit_base_model = ViTModel.from_pretrained("google/vit-base-patch16-224").to(device)
    
    class ViTFeatureExtractor(nn.Module):
        def __init__(self, vit_model):
            super().__init__()
            self.vit = vit_model
            
        def forward(self, x):
            outputs = self.vit(pixel_values=x)
            return outputs.last_hidden_state[:, 0, :]
    
    feature_extractor = ViTFeatureExtractor(vit_base_model)
    INPUT_DIM = 768
    simclr_model = ImprovedSimCLR(feature_extractor, INPUT_DIM, HIDDEN_DIM, PROJECTION_DIM).to(device)

else:
    raise ValueError(f"Backbone {backbone} not supported")

# Optimizer selection
criterion = NTXentLoss(temperature=TEMPERATURE)

if backbone == "vit":
    # Use AdamW for ViT
    optimizer = torch.optim.AdamW(
        simclr_model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999)
    )
else:
    # Use LARS for CNN backbones
    scaled_lr = BASE_LR * (BATCH_SIZE_SIMCLR / 256)
    optimizer = LARS(
        simclr_model.parameters(),
        lr=scaled_lr,
        momentum=0.9,
        weight_decay=WEIGHT_DECAY,
        eta=0.001
    )

# Warmup + cosine scheduler
scheduler = WarmupCosineScheduler(
    optimizer,
    warmup_epochs=WARMUP_EPOCHS,
    total_epochs=NUM_EPOCHS_SIMCLR,
    base_lr=BASE_LR if backbone == "vit" else scaled_lr,
    min_lr=1e-5
)

print(f"\nOptimizer: {'AdamW' if backbone == 'vit' else 'LARS'}")
print(f"Base LR: {BASE_LR}, Warmup: {WARMUP_EPOCHS} epochs")
print(f"Batch size: {BATCH_SIZE_SIMCLR}, Accumulation steps: {ACCUMULATION_STEPS}")

# ============================================
# LOAD DATA 
# ============================================
print(f"\nLoading data from: {DATA_FOLDER}")
if not os.path.exists(DATA_FOLDER):
    raise FileNotFoundError(f"Data folder not found: {DATA_FOLDER}")

data = {}
data_simclr = {}
for folder_label in os.listdir(DATA_FOLDER):
    label_path = os.path.join(DATA_FOLDER, folder_label)
    if not os.path.isdir(label_path):
        continue
    
    parts = folder_label.split('_')
    if len(parts) < 3:
        print(f'Skipping folder with unexpected format: {folder_label}')
        continue
    
    try:
        patient_id = int(parts[1])
        polyp_number = int(parts[2][1:])
    except (IndexError, ValueError) as e:
        print(f'Skipping folder {folder_label}: {e}')
        continue
    
    image_paths = sorted(glob(os.path.join(label_path, '*.jpg')))
       
    
    if len(image_paths) >= 5:
        data.setdefault(patient_id, {}).setdefault(polyp_number, []).extend(image_paths[:5])
      
        # Select image 2, image 3, and image 4
        image_2 = image_paths[1]
        image_3 = image_paths[2]
        image_4 = image_paths[3]
    
        # Add image 2, image 3, and image 4 to the data_simclr dictionary
        data_simclr.setdefault(patient_id, {}).setdefault(polyp_number, []).extend([image_2, image_3, image_4])
    else:
        print(f'Skipping folder {folder_label} for exemplars: Only {len(image_paths)} images found')

print(f"Loaded data for {len(data)} patients")

# ============================================
# DATASET (unchanged)
# ============================================
class PolypDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        image1 = self.transform(image)
        image2 = self.transform(image)
        return image1, image2

# ============================================
# PREPARE DATASETS
# ============================================
all_image_paths = [img for patient in data_simclr.values() for polyp in patient.values() for img in polyp]

def extract_patient_id(image_path):
    folder_name = os.path.basename(os.path.dirname(image_path))
    parts = folder_name.split('_')
    try:
        patient_id = int(parts[1])
        return patient_id
    except (IndexError, ValueError) as e:
        raise ValueError(f"Could not extract patient ID from path: {image_path}")

patient_images = {}
for img_path in all_image_paths:
    patient_id = extract_patient_id(img_path)
    patient_images.setdefault(patient_id, []).append(img_path)

train_patients, val_patients = train_test_split(
    list(patient_images.keys()), 
    test_size=VAL_SPLIT, 
    random_state=RANDOM_STATE
)

train_paths = [img for p in train_patients for img in patient_images[p]]
val_paths = [img for p in val_patients for img in patient_images[p]]

print(f"\nSimCLR Training - Train: {len(train_paths)}, Val: {len(val_paths)}")

train_dataset = PolypDataset(train_paths, simclr_transform)
val_dataset = PolypDataset(val_paths, simclr_transform)

train_dataloader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE_SIMCLR, 
    shuffle=True, 
    drop_last=True,
    num_workers=4,  # Add workers for faster loading
    pin_memory=True
)
val_dataloader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE_SIMCLR, 
    shuffle=False, 
    drop_last=True,
    num_workers=4,
    pin_memory=True
)

# ============================================
# TRAINING LOG
# ============================================
training_log = {
    'backbone': backbone,
    'epochs': [],
    'train_losses': [],
    'val_losses': [],
    'learning_rates': [],
    'best_epoch': None,
    'best_val_loss': None,
    'early_stopped': False,
    'final_epoch': None,
    'hyperparameters': {
        'batch_size': BATCH_SIZE_SIMCLR,
        'base_lr': BASE_LR,
        'weight_decay': WEIGHT_DECAY,
        'temperature': TEMPERATURE,
        'projection_dim': PROJECTION_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_epochs': NUM_EPOCHS_SIMCLR,
        'warmup_epochs': WARMUP_EPOCHS,
        'patience': PATIENCE,
        'optimizer': 'AdamW' if backbone == 'vit' else 'LARS',
        'accumulation_steps': ACCUMULATION_STEPS
    },
    'dataset_info': {
        'train_images': len(train_paths),
        'val_images': len(val_paths),
        'total_images': len(all_image_paths)
    }
}

# ============================================
# TRAINING LOOP
# ============================================
print("\n" + "="*60)
print("TRAINING SIMCLR WITH IMPROVED CONFIGURATION")
print("="*60)

best_val_loss = float('inf')
epochs_without_improvement = 0
best_model_state = None

for epoch in range(NUM_EPOCHS_SIMCLR):
    # Training phase
    simclr_model.train()
    train_loss = 0.0
    train_batches = 0
    
    optimizer.zero_grad()
    
    pbar = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS_SIMCLR} [Train]")
    for batch_idx, (image1, image2) in enumerate(pbar):
        image1, image2 = image1.to(device, non_blocking=True), image2.to(device, non_blocking=True)
        
        z_i = simclr_model(image1)
        z_j = simclr_model(image2)
        
        # Scale loss for gradient accumulation
        loss = criterion(z_i, z_j) / ACCUMULATION_STEPS
        
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"\nWarning: Invalid loss at epoch {epoch + 1}, batch {batch_idx}")
            continue
        
        loss.backward()
        
        # Gradient accumulation
        if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(simclr_model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
        
        train_loss += loss.item() * ACCUMULATION_STEPS
        train_batches += 1
        
        # Update progress bar
        # pbar.set_postfix({'loss': f'{loss.item() * ACCUMULATION_STEPS:.4f}'})
    
    avg_train_loss = train_loss / train_batches if train_batches > 0 else float('inf')
    
    # Validation phase
    simclr_model.eval()
    val_loss = 0.0
    val_batches = 0
    
    with torch.no_grad():
        for image1, image2 in tqdm(val_dataloader, desc=f"Epoch {epoch + 1} [Val]"):
            image1, image2 = image1.to(device), image2.to(device)
            
            z_i = simclr_model(image1)
            z_j = simclr_model(image2)
            loss = criterion(z_i, z_j)
            
            if not (torch.isnan(loss) or torch.isinf(loss)):
                val_loss += loss.item()
                val_batches += 1
    
    avg_val_loss = val_loss / val_batches if val_batches > 0 else float('inf')
    
    # Step scheduler and get current LR
    current_lr = scheduler.step()
    
    # Log metrics
    training_log['epochs'].append(epoch + 1)
    training_log['train_losses'].append(avg_train_loss)
    training_log['val_losses'].append(avg_val_loss)
    training_log['learning_rates'].append(current_lr)
    
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS_SIMCLR} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"LR: {current_lr:.6f}")
    
    # Early stopping check
    if avg_val_loss < (best_val_loss - MIN_DELTA):
        print(f"  ✓ Val loss improved: {best_val_loss:.4f} → {avg_val_loss:.4f}")
        best_val_loss = avg_val_loss
        epochs_without_improvement = 0
        
        training_log['best_epoch'] = epoch + 1
        training_log['best_val_loss'] = best_val_loss
        
        best_model_state = {
            'epoch': epoch + 1,
            'model_state_dict': simclr_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
        }
        
        best_model_path = os.path.join(SAVE_DIR, f'simclr_best_model_{backbone}.pth')
        torch.save(best_model_state, best_model_path)
        print(f"  ✓ Best model saved")
    else:
        epochs_without_improvement += 1
        print(f"  ⚠ No improvement for {epochs_without_improvement}/{PATIENCE} epochs")
        
        if epochs_without_improvement >= PATIENCE:
            print(f"\n{'='*60}")
            print(f"Early stopping at epoch {epoch + 1}!")
            print(f"Best: Epoch {best_model_state['epoch']}, Loss: {best_val_loss:.4f}")
            print(f"{'='*60}")
            training_log['early_stopped'] = True
            training_log['final_epoch'] = epoch + 1
            break

if not training_log['early_stopped']:
    training_log['final_epoch'] = NUM_EPOCHS_SIMCLR

# Save training log
log_filename = os.path.join(SAVE_DIR, f"simclr_training_{backbone}.pkl")
with open(log_filename, 'wb') as f:
    pickle.dump(training_log, f)

print(f"\n{'='*60}")
print(f"Training log saved: {log_filename}")
print("\nSummary:")
print(f"  Total epochs: {training_log['final_epoch']}")
print(f"  Best epoch: {training_log['best_epoch']}")
print(f"  Best val loss: {training_log['best_val_loss']:.4f}")
print(f"  Early stopped: {training_log['early_stopped']}")
print(f"{'='*60}")

# Load best model
if best_model_state:
    simclr_model.load_state_dict(best_model_state['model_state_dict'])
    print(f"\n✓ Loaded best model from epoch {best_model_state['epoch']}")

print("\nSimCLR training complete!")

# ============================================
# EXTRACT EMBEDDINGS 
# ============================================
simclr_model.eval()

extraction_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    image = extraction_transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        embedding = simclr_model.backbone(image)
        embedding = embedding.view(embedding.size(0), -1)
        embedding = F.normalize(embedding, p=2, dim=1)
    
    return embedding.squeeze(0).cpu()


# ============================================
# IMAGE UNIQUENESS VALIDATION FUNCTION
# ============================================
def validate_unique_images(images, exemplar_type, patient_id, polyp_id=None):
    """
    Validate that all image names in the list are unique.
    
    Args:
        images: List of image paths
        exemplar_type: String indicating 'positive' or 'negative'
        patient_id: Patient identifier
        polyp_id: Optional polyp identifier
    
    Raises:
        ValueError: If duplicate images are found
    """
    image_names = [img for img in images]
    if len(image_names) != len(set(image_names)):
        duplicates = [img for img in image_names if image_names.count(img) > 1]
        error_msg = f"Duplicate images found in {exemplar_type} exemplar for patient {patient_id}"
        if polyp_id:
            error_msg += f", polyp {polyp_id}"
        error_msg += f": {set(duplicates)}"
        raise ValueError(error_msg)

# ============================================
# GENERATE EXEMPLARS WITH VALIDATION
# ============================================
print("\n" + "="*50)
print("GENERATING EXEMPLARS WITH UNIQUENESS VALIDATION")
print("="*50)

positive_exemplars = []
positive_patient_ids = []

print("\nGenerating positive exemplars...")
for patient_id, polyps in data.items():
    for polyp_number, images in polyps.items():
        if len(images) >= 5:
            selected_images = images[:5]
            # Validate uniqueness for positive exemplar
            validate_unique_images(selected_images, "positive", patient_id, polyp_number)
            positive_exemplars.append(selected_images)
            positive_patient_ids.append(patient_id)

print(f"✓ Total positive exemplars: {len(positive_exemplars)} (all validated)")

# Generate negative exemplars
patient_ids = list(data.keys())
if len(patient_ids) < 2:
    raise ValueError("Need at least 2 patients to generate negative exemplars")

print("\nGenerating negative exemplars...")
negative_exemplars = []
negative_patient_ids = []
exemplar_errors = [] 
validation_failures = 0
max_retries = 100  # Maximum retries per exemplar to avoid infinite loops

for exemplar_idx in tqdm(range(len(positive_exemplars)), desc="Creating negative exemplars"):
    success = False
    retry_count = 0
    
    while not success and retry_count < max_retries:
        try:
            query_patient_id = random.choice(patient_ids)
            patient_polyps = data[query_patient_id]
            
            # Check if patient has 2 or more polyps
            if len(patient_polyps) >= 2:
                # Patient has 2+ polyps: use different polyp numbers from same patient
                query_polyp_id = random.choice(list(patient_polyps.keys()))
                query_images = patient_polyps[query_polyp_id]
                
                # Select query image from positions 1, 2, or 3 (indices 1, 2, 3)
                valid_query_indices = [i for i in [1, 2, 3] if i < len(query_images)]
                query_image = query_images[random.choice(valid_query_indices)]
                
                # FIXED: Get other polyps and collect all available images
                other_polyp_ids = [pid for pid in patient_polyps.keys() if pid != query_polyp_id]
                available_images = []
                for polyp_id in other_polyp_ids:
                    available_images.extend(patient_polyps[polyp_id])
                
                # Remove query image if it somehow got included
                available_images = [img for img in available_images if img != query_image]
                
                # Check if we have enough unique images
                if len(available_images) < 4:
                    raise ValueError(f"Patient {query_patient_id} has insufficient unique images "
                                   f"(only {len(available_images)} available, need 4)")
                
                # Sample 4 unique images
                target_images = random.sample(available_images, 4)
                
            else:
                # Patient has < 2 polyps: use different patient for target bag
                query_polyp_id = random.choice(list(patient_polyps.keys()))
                query_images = patient_polyps[query_polyp_id]
                
                # Select query image from positions 1, 2, or 3 (indices 1, 2, 3)
                valid_query_indices = [i for i in [1, 2, 3] if i < len(query_images)]
                query_image = query_images[random.choice(valid_query_indices)]
                
                # FIXED: Collect all available images from other patients
                other_patients = [p for p in patient_ids if p != query_patient_id]
                available_images = []
                
                for other_patient_id in other_patients:
                    for other_polyp_id in data[other_patient_id].keys():
                        available_images.extend(data[other_patient_id][other_polyp_id])
                
                # Check if we have enough unique images
                if len(available_images) < 4:
                    raise ValueError(f"Insufficient images available for negative exemplar "
                                   f"(only {len(available_images)} available, need 4)")
                
                # Sample 4 unique images
                target_images = random.sample(available_images, 4)
            
            # Combine and validate uniqueness for negative exemplar
            negative_bag = [query_image] + target_images
            validate_unique_images(negative_bag, "negative", query_patient_id)
            
            # If validation passes, add the exemplar
            negative_exemplars.append(negative_bag)
            negative_patient_ids.append(query_patient_id)
            success = True
            
        except ValueError as e:
            # Validation failed, retry with different selection
            retry_count += 1
            validation_failures += 1

            # Log the error details
            error_entry = {
                'exemplar_index': exemplar_idx + 1,
                'retry_attempt': retry_count,
                'error_type': 'ValueError',
                'error_message': str(e),
                'query_patient_id': query_patient_id,
                'query_polyp_id': query_polyp_id if 'query_polyp_id' in locals() else None,
                'num_patient_polyps': len(patient_polyps),
                'query_image': query_image if 'query_image' in locals() else None,
                'target_images': target_images if 'target_images' in locals() else None,
                'negative_bag': negative_bag if 'negative_bag' in locals() else None
            }
            exemplar_errors.append(error_entry)
            
            if retry_count >= max_retries:
                print(f"\nWarning: Failed to create valid negative exemplar {exemplar_idx + 1} after {max_retries} retries")
                print(f"Error: {e}")
                raise ValueError(f"Could not generate valid negative exemplar after {max_retries} attempts. "
                               f"Dataset may have insufficient unique images.")



# Save only exemplar errors to JSON file
debug_log_path = os.path.join(SAVE_DIR, f'debug_{backbone}.json')
try:
    with open(debug_log_path, 'w') as f:
        json.dump(exemplar_errors, f, indent=2)
    print(f"\n✓ Debug log saved to: {debug_log_path}")
    print(f"  Total error entries: {len(exemplar_errors)}")
except Exception as e:
    print(f"\n✗ Failed to save debug log: {e}")
    
print(f"✓ Total negative exemplars: {len(negative_exemplars)} (all validated)")
if validation_failures > 0:
    print(f"  Note: {validation_failures} validation failures were automatically corrected by resampling")

# Final validation check
print("\n" + "="*50)
print("FINAL VALIDATION CHECK")
print("="*50)

print(f"✓ Positive exemplars: {len(positive_exemplars)} bags, all with unique images")
print(f"✓ Negative exemplars: {len(negative_exemplars)} bags, all with unique images")

# Combine and shuffle
bags = positive_exemplars + negative_exemplars
labels = [1] * len(positive_exemplars) + [0] * len(negative_exemplars)
all_patient_ids = positive_patient_ids + negative_patient_ids

combined = list(zip(bags, labels, all_patient_ids))
random.shuffle(combined)
bags, labels, all_patient_ids = zip(*combined)

print(f"✓ Total exemplars: {len(bags)} (shuffled)")

# Create query-target pairs
query = []
target = []
label_data = []

for bag, label in zip(bags, labels):
    if len(bag) < 5:
        print(f"Warning: Bag has only {len(bag)} images, expected 5")
        continue
    
    # For positive: query from index 1, 2, or 3; target is the rest (4 images)
    # For negative: already structured with query at index 0
    if label == 1:
        query_index = random.choice([1, 2, 3])
    else:
        query_index = 0
    
    target_indices = [i for i in range(len(bag)) if i != query_index]
    query.append(bag[query_index])
    target.append([bag[i] for i in target_indices])
    label_data.append(label)

# ============================================
# EXTRACT EMBEDDINGS
# ============================================
print("\n" + "="*50)
print("EXTRACTING EMBEDDINGS")
print("="*50)

query_embeddings = []
target_embeddings = []
query_paths = []
target_paths = []

print("Extracting query embeddings...")
for query_path in tqdm(query):
    try:
        query_embeddings.append(extract_embedding(query_path))
        query_paths.append(query_path)
    except Exception as e:
        print(f"\nFailed: {query_path}: {e}")

print("Extracting target embeddings...")
for target_bag in tqdm(target):
    try:
        bag_embeddings = [extract_embedding(tp) for tp in target_bag]
        target_embeddings.append(bag_embeddings)
        target_paths.append(target_bag)
    except Exception as e:
        print(f"\nFailed target bag: {e}")

if not query_embeddings:
    raise ValueError("No embeddings extracted. Check image files and model.")

# ============================================
# CREATE SPLITS
# ============================================
print("\n" + "="*50)
print("CREATING SPLITS")
print("="*50)

patient_to_indices = {}
for idx, patient_id in enumerate(all_patient_ids):
    patient_to_indices.setdefault(patient_id, []).append(idx)

unique_patients = list(patient_to_indices.keys())
print(f"Total unique patients: {len(unique_patients)}")

if len(unique_patients) < N_FOLDS:
    print(f"Reducing folds from {N_FOLDS} to {len(unique_patients)}")
    N_FOLDS = len(unique_patients)

random.shuffle(unique_patients)

n_test_patients = max(1, int(len(unique_patients) * TEST_SPLIT))
test_patients = set(unique_patients[:n_test_patients])
trainval_patients = unique_patients[n_test_patients:]

# Create test set
test_idx = [i for i, pid in enumerate(all_patient_ids) if pid in test_patients]
test_labels_array = np.array([label_data[i] for i in test_idx])

test_data = {
    'test_query_embeddings': [query_embeddings[i] for i in test_idx],
    'test_target_embeddings': [target_embeddings[i] for i in test_idx],
    'test_query_paths': [query_paths[i] for i in test_idx],
    'test_target_paths': [target_paths[i] for i in test_idx],
    'test_labels': [label_data[i] for i in test_idx],
    'test_patient_ids': [all_patient_ids[i] for i in test_idx],
    'test_size': len(test_idx),
    'test_class_distribution': {
        'positive': int(np.sum(test_labels_array == 1)), 
        'negative': int(np.sum(test_labels_array == 0))
    },
    'test_patients': sorted(list(test_patients)),
    'n_test_patients': len(test_patients)
}

print(f"\nTest Set: {len(test_idx)} samples, {len(test_patients)} patients")
print(f"  Positive: {test_data['test_class_distribution']['positive']}, "
      f"Negative: {test_data['test_class_distribution']['negative']}")

# Create k-fold splits
patients_per_fold = len(trainval_patients) // N_FOLDS
folds = []

for fold_idx in range(N_FOLDS):
    val_start = fold_idx * patients_per_fold
    val_end = val_start + patients_per_fold if fold_idx < N_FOLDS - 1 else len(trainval_patients)
    
    val_patients = set(trainval_patients[val_start:val_end])
    train_patients = set([p for p in trainval_patients if p not in val_patients])
    
    train_idx = [i for i, pid in enumerate(all_patient_ids) if pid in train_patients]
    val_idx = [i for i, pid in enumerate(all_patient_ids) if pid in val_patients]
    
    train_labels_array = np.array([label_data[i] for i in train_idx])
    val_labels_array = np.array([label_data[i] for i in val_idx])
    
    fold_data = {
        'fold': fold_idx + 1,
        'train_query_embeddings': [query_embeddings[i] for i in train_idx],
        'train_target_embeddings': [target_embeddings[i] for i in train_idx],
        'train_query_paths': [query_paths[i] for i in train_idx],
        'train_target_paths': [target_paths[i] for i in train_idx],
        'train_labels': [label_data[i] for i in train_idx],
        'train_patient_ids': [all_patient_ids[i] for i in train_idx],
        'val_query_embeddings': [query_embeddings[i] for i in val_idx],
        'val_target_embeddings': [target_embeddings[i] for i in val_idx],
        'val_query_paths': [query_paths[i] for i in val_idx],
        'val_target_paths': [target_paths[i] for i in val_idx],
        'val_labels': [label_data[i] for i in val_idx],
        'val_patient_ids': [all_patient_ids[i] for i in val_idx],
        'train_size': len(train_idx),
        'val_size': len(val_idx),
        'train_class_distribution': {
            'positive': int(np.sum(train_labels_array == 1)), 
            'negative': int(np.sum(train_labels_array == 0))
        },
        'val_class_distribution': {
            'positive': int(np.sum(val_labels_array == 1)), 
            'negative': int(np.sum(val_labels_array == 0))
        },
        'train_patients': sorted(list(train_patients)),
        'val_patients': sorted(list(val_patients)),
        'n_train_patients': len(train_patients),
        'n_val_patients': len(val_patients)
    }
    folds.append(fold_data)
    
    print(f"\nFold {fold_idx + 1}:")
    print(f"  Train: {len(train_idx)} samples, {len(train_patients)} patients, "
          f"Pos:{fold_data['train_class_distribution']['positive']}, "
          f"Neg:{fold_data['train_class_distribution']['negative']}")
    print(f"  Val: {len(val_idx)} samples, {len(val_patients)} patients, "
          f"Pos:{fold_data['val_class_distribution']['positive']}, "
          f"Neg:{fold_data['val_class_distribution']['negative']}")
    
    # Validate no patient overlap
    assert len(train_patients & val_patients) == 0
    assert len(train_patients & test_patients) == 0
    assert len(val_patients & test_patients) == 0

# ============================================
# SAVE DATA
# ============================================
print("\n" + "="*50)
print("SAVING DATA")
print("="*50)

os.makedirs(SAVE_DIR, exist_ok=True)

combined_data = {
    'test_data': test_data,
    'folds': folds,
    'n_folds': len(folds),
    'backbone': f'{backbone}_simclr',
    'random_state': RANDOM_STATE,
    'embedding_dim': query_embeddings[0].shape[0],
    'input_dim': INPUT_DIM
}

save_path = os.path.join(SAVE_DIR, f'patient_stratified_test_and_kfold_data_{backbone}_simclr.pkl')

with open(save_path, 'wb') as f:
    pickle.dump(combined_data, f)

print(f"✓ Saved to: {save_path}")
print(f"\nSummary:")
print(f"  Total samples: {len(query_embeddings)}")
print(f"  Test samples: {test_data['test_size']}")
print(f"  Folds: {len(folds)}")
print(f"  Backbone: {backbone}_simclr")
print(f"  Embedding dim: {query_embeddings[0].shape[0]}")
print(f"  Input dim: {INPUT_DIM}")
print(f"  All exemplars validated for unique images: ✓")
print("\n" + "="*50)
print("COMPLETE")
print("="*50)

Using backbone: efficientnet, Device: cuda

Optimizer: LARS
Base LR: 0.3, Warmup: 10 epochs
Batch size: 64, Accumulation steps: 1

Loading data from: /local/puneet/data/Polyps_with_5_images/
Loaded data for 754 patients

SimCLR Training - Train: 4554, Val: 1182

TRAINING SIMCLR WITH IMPROVED CONFIGURATION


Epoch 1 [Val]: 100%|███████████████████████████████████████████████████████████| 18/18 [00:06<00:00,  2.88it/s]


Epoch 1/200 | Train Loss: 4.5329 | Val Loss: 3.9294 | LR: 0.007500
  ✓ Val loss improved: inf → 3.9294
  ✓ Best model saved


Epoch 2 [Val]: 100%|███████████████████████████████████████████████████████████| 18/18 [00:06<00:00,  2.81it/s]


Epoch 2/200 | Train Loss: 4.0204 | Val Loss: 3.8880 | LR: 0.015000
  ✓ Val loss improved: 3.9294 → 3.8880
  ✓ Best model saved


Epoch 3 [Val]: 100%|███████████████████████████████████████████████████████████| 18/18 [00:06<00:00,  2.90it/s]


Epoch 3/200 | Train Loss: 3.8582 | Val Loss: 3.7357 | LR: 0.022500
  ✓ Val loss improved: 3.8880 → 3.7357
  ✓ Best model saved


Epoch 4 [Val]:  67%|███████████████████████████████████████▎                   | 12/18 [00:04<00:01,  3.96it/s]

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score, f1_score
import numpy as np
import os
import pickle
import json
from datetime import datetime

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

"""
Multiple Instance Verification (JMLR 2025) by Xin Xu, Eibe Frank, Geoffrey Holmes
Implementation: https://github.com/xxweka/MIV-head/blob/main/fsc_modeling.py
"""



# ============================================
# ATTENTION MECHANISMS
# ============================================

class NoAttention(nn.Module):
    """Baseline: mean or max pooling without attention."""
    def __init__(self, head_dim, pooling_type='mean'):
        super().__init__()
        self.pooling_type = pooling_type
        
    def forward(self, query, target, target_full=None):
        batch_size, num_queries, _ = query.shape
        num_instances = target.shape[1]
        
        if self.pooling_type == 'mean':
            return torch.ones(batch_size, num_queries, num_instances, device=query.device) / num_instances
        else:  # max
            target_norms = torch.norm(target, dim=-1, keepdim=True)
            max_indices = target_norms.squeeze(-1).argmax(dim=1, keepdim=True)
            attention_weights = torch.zeros(batch_size, num_queries, num_instances, device=query.device)
            attention_weights.scatter_(2, max_indices.unsqueeze(1).expand(-1, num_queries, -1), 1.0)
            return attention_weights


class VEMA(nn.Module):
    """Variance-Enhanced Multi-head Attention."""
    def __init__(self, head_dim, full_dim):
        super().__init__()
        self.sqrt_d = math.sqrt(head_dim)
        self.R = nn.Parameter(torch.randn(full_dim, full_dim))
        self.S = nn.Parameter(torch.randn(full_dim, head_dim))
        nn.init.xavier_uniform_(self.R)
        nn.init.xavier_uniform_(self.S)

    def forward(self, query, target, target_full=None):
        variance_input = target_full if target_full is not None else target
        variance = variance_input.var(dim=1, keepdim=True, unbiased=False) + 1e-8
        centered_variance = variance - 1.0
        gate_input = torch.relu(centered_variance @ self.R)
        delta = torch.sigmoid(gate_input @ self.S)
        gated_query = query * delta
        attention_scores = torch.matmul(gated_query, target.transpose(-2, -1)) / self.sqrt_d
        return F.softmax(attention_scores, dim=-1)


class DBA(nn.Module):
    """Distance-Based Attention."""
    def __init__(self, head_dim, distance_type='l1'):
        super().__init__()
        self.distance_type = distance_type
        self.beta = nn.Parameter(torch.ones(head_dim) * 0.1)

        if distance_type == 'l1':
            self.c = math.sqrt(4.0 / math.pi) * math.sqrt(head_dim)
            self.s = math.sqrt((2.0 - 4.0 / math.pi) * head_dim)
        else:
            self.c = math.sqrt(head_dim)
            self.s = math.sqrt(head_dim / 2.0)
        self.s = max(self.s, 1e-8)

    def forward(self, query, target, target_full=None):
        query_expanded = query.unsqueeze(2)
        target_expanded = target.unsqueeze(1)

        if self.distance_type == 'l1':
            distances = torch.abs(query_expanded - target_expanded)
        else:
            distances = (query_expanded - target_expanded) ** 2

        weighted_distances = distances * (torch.abs(self.beta) + 1e-8)
        distance_sums = weighted_distances.sum(dim=-1)

        if self.distance_type == 'l2':
            distance_sums = torch.sqrt(distance_sums + 1e-8)

        attention_scores = self.c - distance_sums / self.s
        return F.softmax(attention_scores, dim=-1)


class MHSCE(nn.Module):
    """Multi-Head Squeeze-and-Excitation."""
    def __init__(self, embedding_dim, head_dim):
        super().__init__()
        self.J = nn.Parameter(torch.randn(embedding_dim, embedding_dim))
        self.M = nn.Parameter(torch.randn(embedding_dim, head_dim))
        nn.init.xavier_uniform_(self.J)
        nn.init.xavier_uniform_(self.M)

    def forward(self, query):
        mean_query = query.mean(dim=1, keepdim=True)
        excitation_input = torch.relu(mean_query @ self.J)
        return torch.sigmoid(excitation_input @ self.M)


class MultiHeadCrossAttentionPooling(nn.Module):
    """Multi-head cross-attention pooling mechanism."""
    def __init__(self, embedding_dim, num_heads, attention_type='vema'):
        super().__init__()
        assert embedding_dim % num_heads == 0, "embedding_dim must be divisible by num_heads"
        
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads
        self.embedding_dim = embedding_dim
        self.attention_type = attention_type

        self.projection = nn.Linear(embedding_dim, embedding_dim)
        
        if attention_type == 'vema':
            self.attention = VEMA(self.head_dim, embedding_dim)
        elif attention_type in ['dba_l1', 'dba_l2']:
            dist_type = attention_type.split('_')[1]
            self.attention = DBA(self.head_dim, distance_type=dist_type)
        elif attention_type in ['noattention_mean', 'noattention_max']:
            pool_type = attention_type.split('_')[1]
            self.attention = NoAttention(self.head_dim, pooling_type=pool_type)
        else:
            raise ValueError(f"Invalid attention type: {attention_type}")

        self.mhsce = MHSCE(embedding_dim, self.head_dim)
        self.layer_norm_P = nn.LayerNorm(embedding_dim)
        self.layer_norm_Q = nn.LayerNorm(embedding_dim)

    def forward(self, query, target):
        batch_size, num_queries, _ = query.size()
        num_instances = target.size(1)

        # Project and reshape to heads
        query_proj = self.projection(query).view(batch_size, num_queries, self.num_heads, self.head_dim).transpose(1, 2)
        target_proj = self.projection(target).view(batch_size, num_instances, self.num_heads, self.head_dim).transpose(1, 2)

        excitation = self.mhsce(query).unsqueeze(1)

        # Flatten for attention computation
        query_heads_flat = query_proj.reshape(batch_size * self.num_heads, num_queries, self.head_dim)
        target_heads_flat = target_proj.reshape(batch_size * self.num_heads, num_instances, self.head_dim)

        if self.attention_type == 'vema':
            target_full_flat = target.unsqueeze(1).repeat(1, self.num_heads, 1, 1).reshape(
                batch_size * self.num_heads, num_instances, self.embedding_dim)
            attention_weights = self.attention(query_heads_flat, target_heads_flat, target_full_flat)
        else:
            attention_weights = self.attention(query_heads_flat, target_heads_flat)

        attention_weights = attention_weights.view(batch_size, self.num_heads, num_queries, num_instances)
        weighted_sum = torch.matmul(attention_weights, target_proj)

        # Apply excitation and reshape
        co_excited_P = (weighted_sum * excitation).transpose(1, 2).contiguous().view(batch_size, num_queries, self.embedding_dim)
        co_excited_Q = (query_proj * excitation).transpose(1, 2).contiguous().view(batch_size, num_queries, self.embedding_dim)

        return self.layer_norm_P(co_excited_P), self.layer_norm_Q(co_excited_Q)


# ============================================
# SIAMESE NETWORK
# ============================================

class SiameseNetwork(nn.Module):
    def __init__(self, input_dim, embedding_dim, num_heads, attention_type, cap):
        super().__init__()
        assert embedding_dim % num_heads == 0, f"embedding_dim ({embedding_dim}) must be divisible by num_heads ({num_heads})"

        self.shared_fc = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, embedding_dim),
            nn.GroupNorm(num_groups=32, num_channels=embedding_dim)
        )

        self.use_cap = cap
        self.embedding_dim = embedding_dim

        if self.use_cap == 1:
            self.cap = MultiHeadCrossAttentionPooling(embedding_dim, num_heads, attention_type)
            self.alpha = nn.Parameter(torch.ones(embedding_dim))
        else:
            self.similarity_layer = nn.Sequential(
                nn.Linear(embedding_dim * 2, embedding_dim),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(embedding_dim, 1)
            )

    def _process_embedding(self, x):
        """Process embedding with batch norm handling for single samples."""
        if x.dim() == 1:
            x = x.unsqueeze(0)

        if x.size(0) == 1:
            self.eval()
            with torch.no_grad():
                result = self.shared_fc(x)
            self.train()
            return result
        return self.shared_fc(x)

    def baseline_attention_pooling(self, bag_embeddings):
        """Simple attention pooling for baseline."""
        mean_embedding = bag_embeddings.mean(dim=1, keepdim=True)
        attention_scores = torch.matmul(mean_embedding, bag_embeddings.transpose(-2, -1))
        attention_weights = torch.softmax(attention_scores, dim=-1)
        return torch.matmul(attention_weights, bag_embeddings).squeeze(1)

    def forward(self, query, target):
        # Normalize shapes
        if len(target.shape) == 2:
            target = target.unsqueeze(0)
        if len(query.shape) == 1:
            query = query.unsqueeze(0)

        batch_size, num_instances, input_dim = target.shape

        # Embed
        query_embedded = self._process_embedding(query)
        target_embedded = self._process_embedding(target.view(-1, input_dim)).view(batch_size, num_instances, self.embedding_dim)

        if self.use_cap == 1:
            query_embedded = query_embedded.unsqueeze(1)
            vP, vQ = self.cap(query_embedded, target_embedded)
            vP = F.normalize(vP.squeeze(1), p=2, dim=-1)
            vQ = F.normalize(vQ.squeeze(1), p=2, dim=-1)
            similarity_score = torch.sum(vQ * self.alpha * vP, dim=-1, keepdim=True)
        else:
            pooled_target = self.baseline_attention_pooling(target_embedded)
            combined_features = torch.cat([query_embedded, pooled_target], dim=1)
            similarity_score = self.similarity_layer(combined_features)

        return torch.sigmoid(similarity_score)


# ============================================
# DATASET
# ============================================

class QueryTargetDataset(Dataset):
    def __init__(self, query_embeddings, target_embeddings, labels, query_paths, target_paths):
        self.query_embeddings = query_embeddings
        self.target_embeddings = target_embeddings
        self.labels = labels
        self.query_paths = query_paths
        self.target_paths = target_paths

    def __len__(self):
        return len(self.query_embeddings)

    def __getitem__(self, idx):
        query = self.query_embeddings[idx]
        target = torch.stack(self.target_embeddings[idx]) if isinstance(self.target_embeddings[idx], list) else self.target_embeddings[idx]
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return query, target, label, self.query_paths[idx], self.target_paths[idx]


# ============================================
# K-FOLD EXPERIMENTS
# ============================================

class KFoldExperiments:
    def __init__(self, input_dim, embedding_dim, num_heads_list, attention_types, cap, device, batch_size, num_epochs, threshold=0.5, log_dir='training_logs'):
        self.input_dim = input_dim
        self.embedding_dim = embedding_dim
        self.num_heads_list = num_heads_list if isinstance(num_heads_list, list) else [num_heads_list]
        self.attention_types = attention_types
        self.cap = cap
        self.device = device
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.threshold = threshold
        self.log_dir = log_dir
        
        os.makedirs(self.log_dir, exist_ok=True)
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    def create_model(self, attention_type, cap_val, num_heads):
        model = SiameseNetwork(self.input_dim, self.embedding_dim, num_heads, attention_type, cap_val)
        return model.to(self.device)

    def train_fold(self, model, train_loader, val_loader, model_filename, fold_idx, config_key):
        """Train model for one fold using fixed threshold."""
        criterion = nn.BCELoss()
        optimizer = optim.RMSprop(model.parameters(), lr=0.0001)
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer, 
            lambda epoch: 1.0 if epoch < 10 else (0.5 if epoch < 20 else 0.1)
        )

        best_val_loss = float('inf')
        patience_counter = 0
        patience = 10
        
        training_log = {
            'config': config_key,
            'fold': fold_idx + 1,
            'timestamp': self.timestamp,
            'threshold': self.threshold,
            'epochs': []
        }

        for epoch in range(self.num_epochs):
            # Training phase
            model.train()
            train_loss, train_correct, train_total = 0, 0, 0

            for query, target, label, _, _ in train_loader:
                query, target, label = query.to(self.device), target.to(self.device), label.to(self.device)
                
                optimizer.zero_grad()
                output = model(query, target).view(-1)
                label_flat = label.view(-1)
                
                loss = criterion(output, label_flat)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                train_loss += loss.item()
                predicted = (output > self.threshold).float()
                train_correct += (predicted == label_flat).sum().item()
                train_total += label_flat.size(0)

            # Validation phase
            model.eval()
            val_loss = 0
            all_val_labels = []
            all_val_probabilities = []
            all_val_predictions = []

            with torch.no_grad():
                for query, target, label, _, _ in val_loader:
                    query, target, label = query.to(self.device), target.to(self.device), label.to(self.device)
                    
                    output = model(query, target).view(-1)
                    label_flat = label.view(-1)
                    
                    # Collect probabilities and predictions
                    if output.dim() == 0 or output.size(0) == 1:
                        prob = output.cpu().item() if output.dim() == 0 else output.cpu()[0].item()
                        lbl = label_flat.cpu().item() if label_flat.dim() == 0 else label_flat.cpu()[0].item()
                        all_val_probabilities.append(prob)
                        all_val_labels.append(lbl)
                        all_val_predictions.append(1 if prob > self.threshold else 0)
                    else:
                        all_val_probabilities.extend(output.cpu().numpy())
                        all_val_labels.extend(label_flat.cpu().numpy())
                        all_val_predictions.extend((output > self.threshold).float().cpu().numpy())
                    
                    loss = criterion(output, label_flat)
                    val_loss += loss.item()

            scheduler.step()

            # Calculate metrics using fixed threshold
            val_accuracy = 100 * accuracy_score(all_val_labels, all_val_predictions)
            val_f1 = f1_score(all_val_labels, all_val_predictions)
            val_auc = roc_auc_score(all_val_labels, all_val_probabilities)

            # Log epoch stats
            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            train_acc = 100 * train_correct / train_total

            epoch_stats = {
                'epoch': epoch + 1,
                'train_loss': float(avg_train_loss),
                'train_accuracy': float(train_acc),
                'val_loss': float(avg_val_loss),
                'val_accuracy': float(val_accuracy),
                'val_f1': float(val_f1),
                'val_auc': float(val_auc),
                'learning_rate': float(optimizer.param_groups[0]['lr']),
                'is_best': False
            }

            if (epoch + 1) % 5 == 0:
                print(f"Epoch [{epoch + 1}/{self.num_epochs}], Train Loss: {avg_train_loss:.4f}, "
                      f"Val Loss: {avg_val_loss:.4f}, Train Acc: {train_acc:.2f}%, "
                      f"Val Acc: {val_accuracy:.2f}%, Val F1: {val_f1:.4f}")

            # Save best model based on validation loss
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                if os.path.exists(model_filename):
                    os.remove(model_filename)
                torch.save(model.state_dict(), model_filename)
                patience_counter = 0
                epoch_stats['is_best'] = True
                print(f"  ✓ New best model saved! (Epoch {epoch + 1})")
            else:
                patience_counter += 1

            training_log['epochs'].append(epoch_stats)

            # Early stopping
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                training_log['early_stopped'] = True
                training_log['early_stop_epoch'] = epoch + 1
                break
        else:
            training_log['early_stopped'] = False

        # Get best metrics from the best epoch
        best_epoch = max(training_log['epochs'], key=lambda x: x['is_best'])
        
        training_log.update({
            'best_val_loss': float(best_val_loss),
            'best_val_accuracy': float(best_epoch['val_accuracy']),
            'best_val_f1': float(best_epoch['val_f1']),
            'best_val_auc': float(best_epoch['val_auc'])
        })

        print(f"Best validation - Loss: {best_val_loss:.4f}, Acc: {best_epoch['val_accuracy']:.2f}%, "
              f"F1: {best_epoch['val_f1']:.4f}, AUC: {best_epoch['val_auc']:.4f}")

        # Save log
        log_filename = os.path.join(self.log_dir, f'{config_key}_fold_{fold_idx + 1}_{self.timestamp}.json')
        with open(log_filename, 'w') as f:
            json.dump(training_log, f, indent=2)

        return best_val_loss, training_log

    def evaluate_split(self, model, data_loader, split_name="Val"):
        """Evaluate model on a dataset using fixed threshold."""
        model.eval()
        criterion = nn.BCELoss()
        all_labels, all_predictions, all_probabilities = [], [], []
        total_loss = 0

        with torch.no_grad():
            for query, target, label, _, _ in data_loader:
                query, target, label = query.to(self.device), target.to(self.device), label.to(self.device)
                
                output = model(query, target).view(-1)
                label_flat = label.view(-1)
                
                probabilities = output.cpu().numpy() if output.dim() > 0 and output.size(0) > 1 else [output.cpu().item()]
                predicted = (output > self.threshold).float()
                
                all_labels.extend(label_flat.cpu().numpy() if label_flat.dim() > 0 else [label_flat.cpu().item()])
                all_predictions.extend(predicted.cpu().numpy() if predicted.dim() > 0 else [predicted.cpu().item()])
                all_probabilities.extend(probabilities)
                
                total_loss += criterion(output, label_flat).item()

        accuracy = 100 * accuracy_score(all_labels, all_predictions)
        f1 = f1_score(all_labels, all_predictions)
        auc = roc_auc_score(all_labels, all_probabilities)
        cm = confusion_matrix(all_labels, all_predictions, labels=[0, 1])

        return accuracy, f1, total_loss / len(data_loader), cm, auc

    def run_single_kfold_experiment(self, folds, test_data, attention_type, cap_val, num_heads):
        """Run k-fold CV for a single configuration."""
        fold_results = {'accuracies': [], 'f1s': [], 'losses': [], 'cms': [], 'aucs': [], 'logs': []}
        config_key = f"{attention_type}_CAP_{cap_val}_heads_{num_heads}"

        test_dataset = QueryTargetDataset(
            test_data['test_query_embeddings'], test_data['test_target_embeddings'],
            test_data['test_labels'], test_data['test_query_paths'], test_data['test_target_paths']
        )
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size, shuffle=False)

        # Train on each fold
        for fold_idx, fold_data in enumerate(folds):
            print(f"  Fold {fold_idx + 1}/{len(folds)}")

            train_dataset = QueryTargetDataset(
                fold_data['train_query_embeddings'], fold_data['train_target_embeddings'],
                fold_data['train_labels'], fold_data['train_query_paths'], fold_data['train_target_paths']
            )
            val_dataset = QueryTargetDataset(
                fold_data['val_query_embeddings'], fold_data['val_target_embeddings'],
                fold_data['val_labels'], fold_data['val_query_paths'], fold_data['val_target_paths']
            )

            train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False)

            model = self.create_model(attention_type, cap_val, num_heads)
            
            model_dir = f'{backbone}_models'
            os.makedirs(model_dir, exist_ok=True)
            model_filename = os.path.join(model_dir, f'best_model_{config_key}_fold_{fold_idx}.pth')
            
            _, training_log = self.train_fold(
                model, train_loader, val_loader, model_filename, fold_idx, config_key
            )
            fold_results['logs'].append(training_log)

            # Load best model and evaluate
            model.load_state_dict(torch.load(model_filename))
            val_accuracy, val_f1, val_loss, val_cm, val_auc = self.evaluate_split(model, val_loader)
            
            fold_results['accuracies'].append(val_accuracy)
            fold_results['f1s'].append(val_f1)
            fold_results['losses'].append(val_loss)
            fold_results['cms'].append(val_cm)
            fold_results['aucs'].append(val_auc)

            print(f"    Val Acc: {val_accuracy:.2f}%, F1: {val_f1:.4f}, Loss: {val_loss:.4f}, AUC: {val_auc:.4f}")

        # Evaluate best fold on test set
        best_fold_idx = fold_results['accuracies'].index(max(fold_results['accuracies']))
        print(f"\n  Best fold: {best_fold_idx + 1} (Val Acc: {fold_results['accuracies'][best_fold_idx]:.2f}%)")

        best_model_filename = os.path.join(model_dir, f'best_model_{config_key}_fold_{best_fold_idx}.pth')
        best_model = self.create_model(attention_type, cap_val, num_heads)
        best_model.load_state_dict(torch.load(best_model_filename))

        test_accuracy, test_f1, test_loss, test_cm, test_auc = self.evaluate_split(best_model, test_loader, "Test")
        print(f"  Test Acc: {test_accuracy:.2f}%, F1: {test_f1:.4f}, Loss: {test_loss:.4f}, AUC: {test_auc:.4f}")

        # Save summary
        summary = {
            'config': config_key,
            'num_heads': num_heads,
            'threshold': self.threshold,
            'validation': {
                'mean_accuracy': float(np.mean(fold_results['accuracies'])),
                'std_accuracy': float(np.std(fold_results['accuracies'])),
                'best_accuracy': float(max(fold_results['accuracies'])),
                'best_fold': int(best_fold_idx + 1),
                'mean_f1': float(np.mean(fold_results['f1s'])),
                'std_f1': float(np.std(fold_results['f1s'])),
                'mean_loss': float(np.mean(fold_results['losses'])),
                'mean_auc': float(np.mean(fold_results['aucs'])),
                'std_auc': float(np.std(fold_results['aucs']))
            },
            'test': {
                'accuracy': float(test_accuracy),
                'f1': float(test_f1),
                'loss': float(test_loss),
                'auc': float(test_auc)
            }
        }

        with open(os.path.join(self.log_dir, f'{config_key}_summary_{self.timestamp}.json'), 'w') as f:
            json.dump(summary, f, indent=2)

        return {
            'num_heads': num_heads,
            'val_mean_accuracy': summary['validation']['mean_accuracy'],
            'val_std_accuracy': summary['validation']['std_accuracy'],
            'val_best_accuracy': summary['validation']['best_accuracy'],
            'val_best_fold_number': best_fold_idx,
            'val_mean_f1': summary['validation']['mean_f1'],
            'val_std_f1': summary['validation']['std_f1'],
            'val_mean_auc': summary['validation']['mean_auc'],
            'val_std_auc': summary['validation']['std_auc'],
            'test_accuracy': test_accuracy,
            'test_f1': test_f1,
            'test_loss': test_loss,
            'test_auc': test_auc
        }

    def run_all_kfold_experiments(self, data_dict):
        """Run all experiments with different configurations."""
        folds = data_dict['folds']
        test_data = data_dict['test_data']
        all_results = {}

        print(f"\n{'='*80}")
        print(f"K-Fold CV with NUM_HEADS Tuning (Fixed Threshold: {self.threshold})")
        print(f"Folds: {len(folds)}, Test size: {test_data['test_size']}")
        print(f"Attention: {self.attention_types}, CAP: {self.cap}, Heads: {self.num_heads_list}")
        print(f"{'='*80}\n")

        for attention_type in self.attention_types:
            cap_values = [0] if 'noattention' in attention_type else self.cap

            for cap_val in cap_values:
                heads_to_test = [self.num_heads_list[0]] if 'noattention' in attention_type or cap_val == 0 else self.num_heads_list

                for num_heads in heads_to_test:
                    if self.embedding_dim % num_heads != 0:
                        print(f"Skipping {attention_type}, CAP={cap_val}, Heads={num_heads}: embedding_dim not divisible")
                        continue

                    config_key = f"{attention_type}_CAP_{cap_val}_heads_{num_heads}"
                    print(f"\nConfiguration: {config_key}")
                    print("-" * 60)

                    results = self.run_single_kfold_experiment(folds, test_data, attention_type, cap_val, num_heads)
                    all_results[config_key] = results

                    print(f"\n  Validation: {results['val_mean_accuracy']:.2f}% ± {results['val_std_accuracy']:.2f}%, F1: {results['val_mean_f1']:.4f}")
                    print(f"  Test: {results['test_accuracy']:.2f}%, F1: {results['test_f1']:.4f}, AUC: {results['test_auc']:.4f}")

        self.print_final_results(all_results)
        self.save_overall_summary(all_results, folds, test_data)
        return all_results

    def print_final_results(self, all_results):
        """Print comprehensive results summary."""
        print("\n" + "=" * 170)
        print("K-FOLD CROSS VALIDATION RESULTS - VALIDATION SET")
        print("=" * 170)
        print(f"{'Configuration':<35} {'Heads':<8} {'Mean Acc':<18} {'Mean F1':<12} {'Mean AUC':<15} {'Best Fold':<12} {'Best Acc':<12}")
        print("-" * 170)

        for config, results in all_results.items():
            print(f"{config:<35} {results['num_heads']:<8} "
                  f"{results['val_mean_accuracy']:>6.2f}% ± {results['val_std_accuracy']:<5.2f}% "
                  f"{results['val_mean_f1']:>8.4f} "
                  f"{results['val_mean_auc']:>6.4f} "
                  f"{results['val_best_fold_number'] + 1:>6} "
                  f"{results['val_best_accuracy']:>10.2f}%")

        print("\n" + "=" * 170)
        print("HELD-OUT TEST SET RESULTS")
        print("=" * 170)
        print(f"{'Configuration':<35} {'Heads':<8} {'Test Acc':<18} {'Test F1':<12} {'Test AUC':<15}")
        print("-" * 170)

        for config, results in all_results.items():
            print(f"{config:<35} {results['num_heads']:<8} "
                  f"{results['test_accuracy']:>10.2f}% "
                  f"{results['test_f1']:>10.4f} "
                  f"{results['test_auc']:>12.4f}")

        print("\n" + "=" * 170)
        print("BEST CONFIGURATIONS")
        print("=" * 170)

        best_val = max(all_results.items(), key=lambda x: x[1]['val_mean_accuracy'])
        print(f"\nBest Validation Performance: {best_val[0]}")
        print(f"  Heads: {best_val[1]['num_heads']}")
        print(f"  Val Acc: {best_val[1]['val_mean_accuracy']:.2f}% ± {best_val[1]['val_std_accuracy']:.2f}%")
        print(f"  Val F1: {best_val[1]['val_mean_f1']:.4f} ± {best_val[1]['val_std_f1']:.4f}")
        print(f"  Test Acc: {best_val[1]['test_accuracy']:.2f}%, F1: {best_val[1]['test_f1']:.4f}, AUC: {best_val[1]['test_auc']:.4f}")

        best_test = max(all_results.items(), key=lambda x: x[1]['test_accuracy'])
        print(f"\nBest Test Performance: {best_test[0]}")
        print(f"  Heads: {best_test[1]['num_heads']}")
        print(f"  Val Acc: {best_test[1]['val_mean_accuracy']:.2f}% ± {best_test[1]['val_std_accuracy']:.2f}%")
        print(f"  Val F1: {best_test[1]['val_mean_f1']:.4f} ± {best_test[1]['val_std_f1']:.4f}")
        print(f"  Test Acc: {best_test[1]['test_accuracy']:.2f}%, F1: {best_test[1]['test_f1']:.4f}, AUC: {best_test[1]['test_auc']:.4f}")
        
        best_test_f1 = max(all_results.items(), key=lambda x: x[1]['test_f1'])
        print(f"\nBest Test F1 Score: {best_test_f1[0]}")
        print(f"  Heads: {best_test_f1[1]['num_heads']}")
        print(f"  Val Acc: {best_test_f1[1]['val_mean_accuracy']:.2f}% ± {best_test_f1[1]['val_std_accuracy']:.2f}%")
        print(f"  Val F1: {best_test_f1[1]['val_mean_f1']:.4f} ± {best_test_f1[1]['val_std_f1']:.4f}")
        print(f"  Test Acc: {best_test_f1[1]['test_accuracy']:.2f}%, F1: {best_test_f1[1]['test_f1']:.4f}, AUC: {best_test_f1[1]['test_auc']:.4f}")
        print("=" * 170)

    def save_overall_summary(self, all_results, folds, test_data):
        """Save comprehensive experiment summary."""
        summary = {
            'timestamp': self.timestamp,
            'threshold': self.threshold,
            'config': {
                'input_dim': self.input_dim,
                'embedding_dim': self.embedding_dim,
                'num_heads_list': self.num_heads_list,
                'batch_size': self.batch_size,
                'num_epochs': self.num_epochs,
                'attention_types': self.attention_types,
                'cap_values': self.cap
            },
            'data': {
                'n_folds': len(folds),
                'test_size': test_data['test_size'],
                'test_patients': test_data['n_test_patients']
            },
            'results': {
                config: {
                    'num_heads': int(res['num_heads']),
                    'validation': {
                        'mean_accuracy': float(res['val_mean_accuracy']),
                        'std_accuracy': float(res['val_std_accuracy']),
                        'mean_f1': float(res['val_mean_f1']),
                        'std_f1': float(res['val_std_f1']),
                        'mean_auc': float(res['val_mean_auc'])
                    },
                    'test': {
                        'accuracy': float(res['test_accuracy']),
                        'f1': float(res['test_f1']),
                        'auc': float(res['test_auc'])
                    }
                } for config, res in all_results.items()
            }
        }

        filename = os.path.join(self.log_dir, f'overall_summary_{self.timestamp}.json')
        with open(filename, 'w') as f:
            json.dump(summary, f, indent=2)
        print(f"\nOverall summary saved: {filename}")



# Used in the model
EMBEDDING_DIM = 512
NUM_HEADS = [2, 4, 8, 16]
NUM_EPOCHS = 50
BATCH_SIZE = 32
THRESHOLD = 0.5  # Fixed threshold

### Main script
# Load data
print("Loading data...") # vit or res for vit/ res backbones
DATA_PATH = f'/local/kfold_splits/patient_stratified_test_and_kfold_data_{backbone}_simclr.pkl'
with open(DATA_PATH, 'rb') as f:
        kfold_data = pickle.load(f)

print(f"Data loaded successfully!")
print(f"Folds: {kfold_data['n_folds']}, Test size: {kfold_data['test_data']['test_size']}")
print(f"Backbone: {kfold_data['backbone']}")

# Run experiments
experiments = KFoldExperiments(
        input_dim=INPUT_DIM,
        embedding_dim=EMBEDDING_DIM,
        num_heads_list=NUM_HEADS,
        attention_types=['noattention_mean', 'noattention_max', 'vema', 'dba_l1', 'dba_l2'],
        cap=[0, 1],
        device=device,
        batch_size=BATCH_SIZE,
        num_epochs=NUM_EPOCHS,
        threshold=THRESHOLD,
        log_dir=f'training_logs_{backbone}'
    )

results = experiments.run_all_kfold_experiments(kfold_data)

# Save results
save_file_name = f'results_fixed_embedding_{backbone}.pkl'
with open(save_file_name, 'wb') as f:
        pickle.dump(results, f)

print("\n" + "="*80)
print("EXPERIMENT COMPLETE")
print("="*80)
print(f"Logs: {experiments.log_dir}")
print(f"Results saved:{save_file_name}")


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
from PIL import Image

"""
Plot Classification Examples: True Positive, True Negative, False Positive, False Negative
For each configuration, plot multiple examples of each type from the best fold model.
"""



def find_classification_examples(model, test_data, device, threshold=0.5, num_examples=2):
    """
    Find multiple examples each of TP, TN, FP, FN from test data.
    
    Args:
        model: Trained model
        test_data: Test data dictionary
        device: torch device
        threshold: Classification threshold
        num_examples: Number of examples to find for each category
    
    Returns:
        Dictionary with lists of indices and predictions for each type
    """
    query_embeddings = test_data['test_query_embeddings']
    target_embeddings = test_data['test_target_embeddings']
    labels = test_data['test_labels']
    
    # Store examples as lists
    examples = {
        'TP': [],  # True Positive: predicted=1, actual=1
        'TN': [],  # True Negative: predicted=0, actual=0
        'FP': [],  # False Positive: predicted=1, actual=0
        'FN': []   # False Negative: predicted=0, actual=1
    }
    
    model.eval()
    
    with torch.no_grad():
        for idx in range(len(labels)):
            # Skip if we've found all examples
            if all(len(v) >= num_examples for v in examples.values()):
                break
            
            query = query_embeddings[idx].unsqueeze(0).to(device)
            
            if isinstance(target_embeddings[idx], list):
                target = torch.stack(target_embeddings[idx]).unsqueeze(0).to(device)
            else:
                target = target_embeddings[idx].unsqueeze(0).to(device)
            
            label = labels[idx]
            
            # Get prediction
            output = model(query, target)
            pred_score = output.item()
            pred_class = 1 if pred_score >= threshold else 0
            
            # Categorize
            if pred_class == 1 and label == 1 and len(examples['TP']) < num_examples:
                examples['TP'].append({'idx': idx, 'score': pred_score, 'label': label})
            elif pred_class == 0 and label == 0 and len(examples['TN']) < num_examples:
                examples['TN'].append({'idx': idx, 'score': pred_score, 'label': label})
            elif pred_class == 1 and label == 0 and len(examples['FP']) < num_examples:
                examples['FP'].append({'idx': idx, 'score': pred_score, 'label': label})
            elif pred_class == 0 and label == 1 and len(examples['FN']) < num_examples:
                examples['FN'].append({'idx': idx, 'score': pred_score, 'label': label})
    
    return examples


def load_and_display_images(test_data, idx, save_path, example_type, config_key, pred_score):
    """
    Load query and target images, concatenate all horizontally (query leftmost).
    
    Args:
        test_data: Test data dictionary with image paths
        idx: Index of the example
        save_path: Path to save the figure
        example_type: 'TP', 'TN', 'FP', or 'FN'
        config_key: Configuration name
        pred_score: Prediction score
    """
    # Get image paths
    query_path = test_data['test_query_paths'][idx]
    target_paths = test_data['test_target_paths'][idx]
    label = test_data['test_labels'][idx]
    
    # Load images
    query_img = Image.open(query_path).convert('RGB')
    target_imgs = [Image.open(path).convert('RGB') for path in target_paths]
    
    
    # Resize for consistent display
    target_size = (224, 224)
    query_img = query_img.resize(target_size)
    target_imgs = [img.resize(target_size) for img in target_imgs]
    
    # Concatenate ALL images horizontally: query (leftmost) + all targets
    all_imgs = [query_img] + target_imgs
    total_width = sum(img.width for img in all_imgs)
    max_height = max(img.height for img in all_imgs)
    concatenated_all = Image.new('RGB', (total_width, max_height))
    
    x_offset = 0
    for img in all_imgs:
        concatenated_all.paste(img, (x_offset, 0))
        x_offset += img.width
    
    # Create figure
    fig, ax = plt.subplots(1, 1, figsize=(16, 5))
    
    # Plot concatenated image
    ax.imshow(concatenated_all)
    ax.axis('off')
    
    # Add text labels below the image
    # Query label
    ax.text(query_img.width / 2, max_height + 20, 'Query', 
            ha='center', va='top', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.7))
    
    # Target label
    target_center_x = query_img.width + (total_width - query_img.width) / 2
    ax.text(target_center_x, max_height + 20, f'Targets (n={len(target_imgs)})', 
            ha='center', va='top', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgreen', alpha=0.7))
    
    # Add overall title with prediction info
    type_names = {
        'TP': 'True Positive',
        'TN': 'True Negative',
        'FP': 'False Positive',
        'FN': 'False Negative'
    }
    
    pred_class = 1 if pred_score >= 0.5 else 0
    ax.set_title(
        f'{type_names[example_type]} - {config_key}\n'
        f'Prediction: {pred_class} (score: {pred_score:.3f}) | True Label: {label}',
        fontsize=16, fontweight='bold', pad=20
    )
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"      Saved: {save_path}")
    plt.close()


def plot_examples_for_configuration(combined_data, all_results, config_key, 
                                    model_class, input_dim, embedding_dim, 
                                    device, save_dir='results_examples',
                                    num_examples=2):
    """
    Plot multiple TP, TN, FP, FN examples for a single configuration.
    
    MODIFIED: Extracts num_heads from config_key automatically
    
    Args:
        num_examples: Number of examples to plot for each category (default=2)
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Parse config key: attention_type_CAP_X_heads_Y
    parts = config_key.split('_heads_')
    if len(parts) != 2:
        print(f"Skipping invalid config format: {config_key}")
        return
    
    base_config = parts[0]
    num_heads = int(parts[1])
    
    # Parse base_config: attention_type_CAP_X
    base_parts = base_config.rsplit('_CAP_', 1)
    if len(base_parts) != 2:
        print(f"Skipping invalid config format: {config_key}")
        return
    
    attention_type = base_parts[0]
    cap_val = int(base_parts[1])
    
    # Validate noattention configurations
    if 'noattention' in attention_type and cap_val != 0:
        print(f"Skipping {config_key} - noattention methods only support CAP=0\n")
        return
    
    print(f"\n{'='*80}")
    print(f"Processing: {config_key}")
    print(f"{'='*80}")
    
    # Get best fold
    if config_key not in all_results:
        print(f"Configuration {config_key} not found in results")
        return
    
    fold_results = all_results[config_key]
    best_fold_idx = fold_results['val_best_fold_number']
    test_data = combined_data['test_data']
    
    print(f"Attention Type: {attention_type}")
    print(f"CAP: {cap_val}")
    print(f"Num Heads: {num_heads}")
    print(f"Best Fold: {best_fold_idx + 1}")
    print(f"Test Accuracy: {fold_results['test_accuracy']:.2f}%")
    
    # Load model with correct num_heads
    model = model_class(
        input_dim=input_dim,
        embedding_dim=embedding_dim,
        num_heads=num_heads,
        attention_type=attention_type,
        cap=cap_val
    ).to(device)
    
    # Model filename now includes num_heads
    model_filename = f'{backbone}_models/best_model_{attention_type}_CAP_{cap_val}_heads_{num_heads}_fold_{best_fold_idx}.pth'
    
    if not os.path.exists(model_filename):
        print(f"Model file not found: {model_filename}")
        return
    
    model.load_state_dict(torch.load(model_filename, map_location=device))
    model.eval()
    
    print(f"Loaded model from: {model_filename}")
    
    # Find examples
    print(f"Finding {num_examples} classification examples for each category...")
    examples = find_classification_examples(model, test_data, device, num_examples=num_examples)
    
    # Plot each example
    for example_type, example_list in examples.items():
        if len(example_list) == 0:
            print(f"  - {example_type}: No examples found")
            continue
        
        print(f"  - {example_type}: Found {len(example_list)} example(s)")
        
        for i, example_data in enumerate(example_list):
            idx = example_data['idx']
            pred_score = example_data['score']
            
            # Add example number to filename
            save_path = os.path.join(save_dir, f'{config_key}_{example_type}_{i+1}.png')
            
            try:
                load_and_display_images(
                    test_data=test_data,
                    idx=idx,
                    save_path=save_path,
                    example_type=example_type,
                    config_key=config_key,
                    pred_score=pred_score
                )
                print(f"      Example {i+1}: index {idx} (score: {pred_score:.3f})")
            except Exception as e:
                print(f"      Example {i+1}: Error - {str(e)}")
                import traceback
                traceback.print_exc()


def plot_all_configurations(combined_data, all_results, model_class,
                            input_dim, embedding_dim, device,
                            save_dir='results_examples', num_examples=2):
    """
    Plot multiple TP, TN, FP, FN examples for all configurations.
    
    MODIFIED: Removed num_heads parameter - now extracted from config_key
    
    Args:
        num_examples: Number of examples to plot for each category (default=2)
    """
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"PLOTTING {num_examples} CLASSIFICATION EXAMPLES PER CATEGORY FOR ALL CONFIGURATIONS")
    print(f"Saving to: {save_dir}/")
    print(f"{'='*80}\n")
    
    successful = 0
    failed = 0
    
    for config_key in sorted(all_results.keys()):
        try:
            plot_examples_for_configuration(
                combined_data=combined_data,
                all_results=all_results,
                config_key=config_key,
                model_class=model_class,
                input_dim=input_dim,
                embedding_dim=embedding_dim,
                device=device,
                save_dir=save_dir,
                num_examples=num_examples
            )
            successful += 1
        except Exception as e:
            print(f"Error processing {config_key}: {str(e)}\n")
            import traceback
            traceback.print_exc()
            failed += 1
            continue
    
    print(f"\n{'='*80}")
    print(f"SUMMARY")
    print(f"{'='*80}")
    print(f"Successfully processed: {successful} configurations")
    print(f"Failed: {failed} configurations")
    print(f"Results saved to: {save_dir}/")
    print(f"Expected {num_examples} examples per category (TP, TN, FP, FN) per configuration")
    print(f"{'='*80}\n")


def plot_specific_configurations(combined_data, all_results, model_class,
                                 input_dim, embedding_dim, device,
                                 config_list, save_dir='results_examples',
                                 num_examples=2):
    """
    Plot examples for specific configurations only.
    
    Args:
        config_list: List of config keys to plot (e.g., ['vema_CAP_1_heads_8'])
        num_examples: Number of examples per category
    """
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"PLOTTING SPECIFIC CONFIGURATIONS")
    print(f"Configurations to plot: {len(config_list)}")
    print(f"{'='*80}\n")
    
    for config_key in config_list:
        if config_key not in all_results:
            print(f"Warning: {config_key} not found in results\n")
            continue
        
        try:
            plot_examples_for_configuration(
                combined_data=combined_data,
                all_results=all_results,
                config_key=config_key,
                model_class=model_class,
                input_dim=input_dim,
                embedding_dim=embedding_dim,
                device=device,
                save_dir=save_dir,
                num_examples=num_examples
            )
        except Exception as e:
            print(f"Error processing {config_key}: {str(e)}\n")
            import traceback
            traceback.print_exc()


def summarize_examples(save_dir='results_examples'):
    """
    Summarize what examples were generated.
    """
    if not os.path.exists(save_dir):
        print(f"Directory {save_dir} does not exist")
        return
    
    files = [f for f in os.listdir(save_dir) if f.endswith('.png')]
    
    if not files:
        print(f"No PNG files found in {save_dir}")
        return
    
    # Group by configuration
    configs = {}
    for f in files:
        # Parse filename: config_key_ExampleType_N.png
        parts = f.rsplit('_', 2)
        if len(parts) == 3:
            config = parts[0]
            example_type = parts[1]
            
            if config not in configs:
                configs[config] = {'TP': 0, 'TN': 0, 'FP': 0, 'FN': 0}
            
            if example_type in configs[config]:
                configs[config][example_type] += 1
    
    print(f"\n{'='*80}")
    print(f"SUMMARY OF GENERATED EXAMPLES")
    print(f"Directory: {save_dir}")
    print(f"{'='*80}\n")
    print(f"Total files: {len(files)}")
    print(f"Configurations: {len(configs)}\n")
    
    for config, counts in sorted(configs.items()):
        print(f"{config}:")
        print(f"  TP: {counts['TP']}, TN: {counts['TN']}, "
              f"FP: {counts['FP']}, FN: {counts['FN']}")
    
    print(f"\n{'='*80}\n")


save_file_directory = f"results_examples_{backbone}"
plot_all_configurations(combined_data=combined_data, all_results=results,
                            model_class=SiameseNetwork, input_dim=INPUT_DIM, embedding_dim=512, device=device,
                            save_dir=save_file_directory, num_examples=2)
                            